# ClassicPitch

## 1. Load Model

In [ ]:
from predictor import ClassicPitch

predictor = ClassicPitch(device='mps', merge_notes=True)

## 2. Load Audio

In [ ]:
import IPython.display

fs = 22050
audio_file = "data/goodbye.wav"
IPython.display.Audio(audio_file, rate=fs)

Or download from YouTube.

In [ ]:
yt_link = "https://www.youtube.com/watch?v=KwIC6B_dvW4"
audio_file = predictor.get_audio_from_youtube(yt_link)
IPython.display.Audio(audio_file, rate=fs)

## 3. Transcribe

In [ ]:
midi, model_output, times = predictor.predict(audio_file, verbose=True)
midi.write("data/output.mid")

## 4. Sonify

In [ ]:
IPython.display.Audio(midi.synthesize(fs=fs), rate=fs)

Transcription left, original right.

In [ ]:
import numpy as np
import librosa

transcription = midi.synthesize(fs=fs)
original, _ = librosa.load(audio_file, sr=fs, mono=True)
n = min(len(transcription), len(original))
mix = np.stack([transcription[:n], original[:n] * 0.3])
IPython.display.Audio(mix / np.abs(mix).max(), rate=fs)

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

start, end = 5, 25
extent = [times[0], times[-1], 24, 96]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].imshow(midi.get_piano_roll(fs=100)[24:96] > 0, aspect='auto', origin='lower',
               interpolation='none', extent=[0, midi.get_end_time(), 24, 96], cmap='Greys')
axes[0].set_title('Transcription')
axes[1].imshow(model_output['note'], aspect='auto', origin='lower', interpolation='none', extent=extent, cmap='Greys')
axes[1].set_title('Frame activation')
axes[2].imshow(model_output['onset'], aspect='auto', origin='lower', interpolation='none', extent=extent, cmap='Greys')
axes[2].set_title('Onset activation')
axes[2].set_xlim(start, end)
axes[2].set_xlabel('Time (s)')
plt.show()